In [36]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import numpy as np
from torch.utils.tensorboard import SummaryWriter

In [37]:
with open('girls-names.csv') as f:
    content = f.readlines()
print(len(content))
content = ' '.join( [c.strip()+'.' for c in content])
content[:100]

# allowed_chars = set(string.ascii_letters + string.digits + ". ")

# with open('persuasion.txt') as f:
#     content = f.read()
# print(len(content))
# content = ''.join([c for c in content if c in allowed_chars])
# content[:100]

3759


'Aadhya. Aadya. Aaira. Aairah. Aaliyah. Aanya. Aaradhya. Aarna. Aarvi. Aarya. Aashvi. Aavya. Aayat. A'

In [38]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(device)

mps


In [39]:
allowed_chars = set(string.ascii_letters + string.digits + ". ")

content = ''.join([c for c in content if c in allowed_chars])

chars = set(sorted(list(content)))
i_to_c = {k:v for k, v in enumerate(chars)}
c_to_i = {k:v for v, k in i_to_c.items()}

encode = lambda s: [c_to_i[c] for c in s] # noqa: E731
decode = lambda s: ''.join([i_to_c[i] for i in s]) # noqa: E731
decode(encode(content[:100]))

encoded_content = torch.tensor(encode(content))

In [40]:
context_length = 4

class NameGenerator(nn.Module):
    def __init__(self, vocab_size, context_length, hidden_dim):
        super().__init__()
        
        input_dim = context_length * vocab_size
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=vocab_size)

        self.net = nn.Sequential(    
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
            )

    def forward(self, x):
        emb = self.embedding(x)
        emb = emb.view(emb.shape[0], -1)
        logits = self.net(emb)
        return logits

model = NameGenerator(len(chars), context_length, 512)
optimizer = torch.optim.Adam(model.parameters(), lr=.01)

In [41]:
from torch.utils.data import TensorDataset, DataLoader

xs = []
ys = []
for i in range(0, len(encoded_content)-context_length):
    x_chunk = encoded_content[i:i+context_length]
    y_chunk = encoded_content[i+context_length]

    xs.append(x_chunk)
    ys.append(y_chunk)

X = torch.stack(xs)
Y = torch.stack(ys)

dataset = TensorDataset(X, Y)

loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [42]:
for xb, yb in loader:
    logits = model.forward(xb)
    
    loss = F.cross_entropy(logits.view(-1, len(chars)), yb)
    print(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

3.9990572929382324
3.816222906112671
4.5413336753845215
2.960204839706421
2.9795279502868652
2.8731205463409424
2.5902812480926514
2.4372072219848633
3.088088035583496
3.400677442550659
2.653003454208374
2.5278549194335938
2.891786575317383
2.18052077293396
2.5619401931762695
2.6379802227020264
2.9933319091796875
2.6235451698303223
1.9116052389144897
2.0628654956817627
2.79133939743042
2.533763885498047
2.619751453399658
3.341341733932495
2.739701271057129
2.678866386413574
2.0108792781829834
2.1617722511291504
2.3751325607299805
2.796349048614502
3.0977253913879395
2.1720352172851562
1.9999785423278809
2.7924816608428955
2.249650478363037
3.1452765464782715
1.9453585147857666
2.395301342010498
2.2897088527679443
1.9653115272521973
3.0586328506469727
2.6233136653900146
2.1857030391693115
2.3413009643554688
2.454245090484619
2.7283952236175537
2.3204681873321533
2.2833142280578613
2.5290744304656982
2.700256586074829
2.639291524887085
2.1204628944396973
2.3795204162597656
2.597093582153

In [43]:
# Girls names
for initial in [ch for ch in string.ascii_letters if ch==ch.upper()]:
    inputs = ' ' * (context_length-1) + initial

    for i in range(15):
        output = decode([model.forward(torch.tensor(encode(inputs[-context_length:])).view(1,-1)).argmax().item()])
        inputs+=output
        if output=='.': break
    print(inputs)


   Ana.
   Bie.
   Cerin.
   Deria.
   Elia.
   Fielie.
   Gia.
   Herie.
   Inea.
   Jeilie.
   Keilie.
   Leina.
   Merie.
   Neille.
   Orien.
   Piena.
   Qeen.
   Reee.
   Sanee.
   Teina.
   Une.
   Virea.
   Win.
   Xerie.
   Yana.
   Zerie.


In [44]:
# Infinite persuasion

# inputs = ' ' * (context_length-1) + 'Z'

# for i in range(1500):
#     output = decode([model.forward(torch.tensor(encode(inputs[-context_length:])).view(1,-1)).argmax().item()])
#     inputs+=output
#     # if output=='.': break
# print(inputs+'.')
